<a href="https://colab.research.google.com/github/pravinjaju1005/python/blob/main/NSE_PE_PB_Div_to_Gsheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-requisites

## Installing and Importing required Libraries

In [1]:
!pip install -q nsepython

In [2]:
from nsepython import index_pe_pb_div
from datetime import datetime
import pandas as pd

## Authentications for connecting to Google Sheets

In [14]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# NSE TRI Data Extraction

In [3]:
# Getting all Index Values in a List

def fetch_index_symbols(date: str):
    url = f"https://archives.nseindia.com/content/indices/ind_close_all_{date}.csv"
    try:
        df = pd.read_csv(url)
        # Extract unique index symbols
        symbols = df['Index Name'].dropna().unique().tolist()
        return symbols
    except Exception as e:
        print(f"❌ Failed to fetch for {date}: {e}")
        return []

# Example usage
date = "01082025"  # 1st Jan 2025
index_symbols = fetch_index_symbols(date)

In [7]:
# Getting TRI Values for all the index

start_str = "2025-08-01"
# end_str   = "2025-06-27"
end_str   = datetime.now().strftime('%Y-%m-%d')

start_date_formatted = datetime.strptime(start_str, '%Y-%m-%d').strftime("%d-%b-%Y")
end_date_formatted = datetime.strptime(end_str, '%Y-%m-%d').strftime("%d-%b-%Y")

# Store DataFrames
all_pe_pb_div_dfs = []

# Fetch data for each index
for index_sym in index_symbols:
    try:
        df_index = index_pe_pb_div(index_sym, start_date_formatted, end_date_formatted)
        if df_index is not None and not df_index.empty:
            all_pe_pb_div_dfs.append(df_index)
    except Exception as e:
        print(f"Failed for {index_sym}: {e}")
        continue

# Combine data
if not all_pe_pb_div_dfs:
    print("No data could be fetched for any index.")
else:
    combined_pe_pb_div_df = pd.concat(all_pe_pb_div_dfs, ignore_index=True)

    # Clean and convert columns
    combined_pe_pb_div_df['DATE'] = pd.to_datetime(combined_pe_pb_div_df['DATE'], errors='coerce')
    combined_pe_pb_div_df['pe'] = pd.to_numeric(combined_pe_pb_div_df['pe'], errors='coerce')
    combined_pe_pb_div_df['pb'] = pd.to_numeric(combined_pe_pb_div_df['pb'], errors='coerce')
    combined_pe_pb_div_df['divYield'] = pd.to_numeric(combined_pe_pb_div_df['divYield'], errors='coerce')

    # Drop missing rows
    combined_pe_pb_div_df.dropna(subset=['DATE', 'pe', 'pb', 'divYield'], inplace=True)

    # Pivot data
    pe_df = combined_pe_pb_div_df.pivot(index='DATE', columns='Index Name', values='pe')
    pb_df = combined_pe_pb_div_df.pivot(index='DATE', columns='Index Name', values='pb')
    div_yield_df = combined_pe_pb_div_df.pivot(index='DATE', columns='Index Name', values='divYield')

    # Sort
    pe_df.sort_index(ascending=False, inplace=True)
    pb_df.sort_index(ascending=False, inplace=True)
    div_yield_df.sort_index(ascending=False, inplace=True)

In [17]:
pe_df.head()

Index Name,NIFTY Midcap 100,NIFTY SME EMERGE,NIFTY100 Alpha 30,NIFTY100 ESG,NIFTY200 Quality 30,NIFTY500 Value 50,Nifty 100,Nifty 200,Nifty 50,Nifty 500,...,Nifty Realty,Nifty Rural,Nifty Shariah 25,Nifty200 Alpha 30,Nifty200 Value 30,Nifty50 Shariah,Nifty50 Value 20,Nifty500 Flexicap Quality 30,Nifty500 Healthcare,Nifty500 Shariah
DATE,,,,,,,,,,,,,,,,,,,,,
2025-08-18,31.07,26.07,37.83,22.48,28.02,10.43,21.71,22.87,21.86,24.02,...,43.08,24.95,26.26,54.38,9.28,23.54,16.57,30.33,39.43,28.28
2025-08-14,30.84,25.79,37.44,22.28,28.33,10.48,21.56,22.71,21.66,23.83,...,42.54,24.40,26.07,54.69,9.38,23.43,16.49,29.95,39.56,28.29
2025-08-13,30.96,25.83,37.29,22.25,28.40,10.60,21.58,22.74,21.68,23.87,...,42.86,24.40,26.12,54.41,9.49,23.43,16.50,29.82,39.61,28.33
2025-08-12,30.75,25.81,36.82,22.13,28.26,10.53,21.46,22.61,21.56,23.73,...,42.76,24.30,25.91,53.75,9.44,23.24,16.43,29.69,38.90,28.16
2025-08-11,30.78,25.70,37.00,22.20,28.30,10.51,21.56,22.70,21.67,23.80,...,43.07,24.42,25.89,54.10,9.45,23.14,16.49,29.66,38.73,28.07


In [9]:
# Reset index for all three pivoted DataFrames
pe_df_reset = pe_df.reset_index()
pb_df_reset = pb_df.reset_index()
div_yield_df_reset = div_yield_df.reset_index()

# Melt each DataFrame to long format
pe_transformed_df = pe_df_reset.melt(id_vars=['DATE'], var_name='Index Name', value_name='PE')
pb_transformed_df = pb_df_reset.melt(id_vars=['DATE'], var_name='Index Name', value_name='PB')
div_yield_transformed_df = div_yield_df_reset.melt(id_vars=['DATE'], var_name='Index Name', value_name='DivYield')

# Sort each by date (ascending for clean visualization)
pe_transformed_df.sort_values(by='DATE', ascending=True, inplace=True)
pb_transformed_df.sort_values(by='DATE', ascending=True, inplace=True)
div_yield_transformed_df.sort_values(by='DATE', ascending=True, inplace=True)

# Extracting Data to Google Sheet

In [18]:
from gspread_dataframe import set_with_dataframe

# Spreadsheet name
spreadsheet_name = 'NSE PE PB DY Data 2025'

# List of sheets and corresponding DataFrames
data_exports = {
    'PE Data': pe_df.reset_index().assign(DATE=lambda df: df['DATE'].dt.strftime('%Y-%m-%d')),
    'PB Data': pb_df.reset_index().assign(DATE=lambda df: df['DATE'].dt.strftime('%Y-%m-%d')),
    'Div Yield Data': div_yield_df.reset_index().assign(DATE=lambda df: df['DATE'].dt.strftime('%Y-%m-%d'))
}

try:
    sh = gc.open(spreadsheet_name)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Spreadsheet '{spreadsheet_name}' not found. Creating it.")
    sh = gc.create(spreadsheet_name)

# Loop through each sheet/data pair
for sheet_name, df in data_exports.items():
    try:
        worksheet = sh.worksheet(sheet_name)
    except gspread.exceptions.WorksheetNotFound:
        print(f"Worksheet '{sheet_name}' not found. Creating it.")
        worksheet = sh.add_worksheet(sheet_name, rows="100", cols="20")

    # Clear existing content
    worksheet.clear()
    print(f"Worksheet '{sheet_name}' cleared.")

    # Upload DataFrame
    set_with_dataframe(worksheet, df, include_index=False, include_column_header=True)
    print(f"Data exported to: https://docs.google.com/spreadsheets/d/{sh.id}/edit#gid={worksheet.id}")


Worksheet 'PE Data' cleared.
Data exported to: https://docs.google.com/spreadsheets/d/1BnKmZFEinrAF_9iLFySjBqOgB5pA-HOuOvGYNJPnVp8/edit#gid=156987288
Worksheet 'PB Data' cleared.
Data exported to: https://docs.google.com/spreadsheets/d/1BnKmZFEinrAF_9iLFySjBqOgB5pA-HOuOvGYNJPnVp8/edit#gid=645082688
Worksheet 'Div Yield Data' cleared.
Data exported to: https://docs.google.com/spreadsheets/d/1BnKmZFEinrAF_9iLFySjBqOgB5pA-HOuOvGYNJPnVp8/edit#gid=1661032218


In [ ]:
# # Data for viewing
# spreadsheet_name = 'NSE TRI Data 2025' # Replace with your sheet's name
# worksheet_name = 'Sheet1' # Replace with your desired worksheet name

# try:
#     sh = gc.open(spreadsheet_name)
#     worksheet = sh.worksheet(worksheet_name)
# except gspread.exceptions.SpreadsheetNotFound:
#     print(f"Spreadsheet '{spreadsheet_name}' not found. Creating a new spreadsheet.")
#     sh = gc.create(spreadsheet_name)
#     # When you create a new spreadsheet, it automatically has a 'Sheet1'.
#     # We'll try to get it, or add if for some reason it's not named 'Sheet1' or we want a different name.
#     try:
#         worksheet = sh.worksheet(worksheet_name)
#     except gspread.exceptions.WorksheetNotFound:
#         print(f"Worksheet '{worksheet_name}' not found in new spreadsheet. Creating it.")
#         worksheet = sh.add_worksheet(worksheet_name, rows="100", cols="20") # Adjust rows/cols as needed
# except gspread.exceptions.WorksheetNotFound:
#     print(f"Worksheet '{worksheet_name}' not found in '{spreadsheet_name}'. Creating new worksheet.")
#     worksheet = sh.add_worksheet(worksheet_name, rows="100", cols="20") # Adjust rows/cols as needed

# # --- ADD THIS LINE TO CLEAR THE SHEET ---
# worksheet.clear()
# print(f"Worksheet '{worksheet_name}' in '{spreadsheet_name}' cleared.")
# # --- END OF ADDITION ---

# from gspread_dataframe import set_with_dataframe
# set_with_dataframe(worksheet, pivoted_df, include_index=True, include_column_header=True) # Set include_index=True if you want to export the DataFrame index

# print(f"Data exported to: https://docs.google.com/spreadsheets/d/{sh.id}/edit#gid={worksheet.id}")